# Master Preprocessing Notebook

Combined preprocessing pipeline for all three models. Run top-to-bottom once to produce all files needed by the model notebooks.

**Outputs produced and where they are consumed:**

| File | Consumed by |
|------|-------------|
| `preprocessing_data/baseline_train_engineered.csv` | `4a_Baseline.ipynb` |
| `preprocessing_data/baseline_test_engineered.csv` | `4a_Baseline.ipynb` |
| `preprocessing_data/X_train_sparse_rf.npz` | `4b_RandomForest_Model.ipynb` (sparse section) |
| `preprocessing_data/X_test_sparse_rf.npz` | `4b_RandomForest_Model.ipynb` (sparse section) |
| `preprocessing_data/y_train_rf.csv` | `4b_RandomForest_Model.ipynb` (sparse section) |
| `preprocessing_data/y_test_rf.csv` | `4b_RandomForest_Model.ipynb` (sparse section) |
| `preprocessing_data/rf_preprocessor.joblib` | `4b_RandomForest_Model.ipynb` (sparse section) |
| `preprocessing_data/train_engineered.csv` | `4b_RandomForest_Model.ipynb` (engineered sections) · `4c_XGBoost_Model.ipynb` |
| `preprocessing_data/test_engineered.csv` | `4b_RandomForest_Model.ipynb` (engineered sections) · `4c_XGBoost_Model.ipynb` |
| `preprocessing_data/feature_engineering_metadata.json` | `4b_RandomForest_Model.ipynb` · `4c_XGBoost_Model.ipynb` |

> **Note on `4b_RandomForest_Model` sparse section:** The sparse matrices are built from
> `train_data_final_feat_no_preprocessing.csv` / `test_data_final_feat_no_preprocessing.csv`,
> which are intermediate files produced by an earlier notebook. Section 2 of this master notebook reads those files and produces the sparse outputs.

> **Note on `4c_XGBoost_Model` cell 44:** That cell uses `COW` and `SCH` columns from
> `train_engineered.csv`. Those columns are added to the `usecols` list in Section 3 below
> so they are available in the saved CSV.

---
---

# SECTION 1 — Logistic Regression (Baseline) Preprocessing

Loads raw data, applies shared feature engineering, label-encodes categoricals, downsamples majority class, and saves `baseline_train/test_engineered.csv`.

**Consumed by:** `4a_Baseline.ipynb`

---
---

In [ ]:
import pandas as pd
from sklearn.utils import resample
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

## 1) Load Raw Data

In [ ]:
df_raw = pd.read_csv('../1_Raw_Data/data_persons_ca_1yr/persons_master.csv')
print(f'Raw data shape: {df_raw.shape}')

df = df_raw[df_raw['AGEP'] > 18].copy()
print(f'Adults only: {df.shape}')

## 2) Target Variable

In [ ]:
def categorize_poverty_score(val):
    if pd.isna(val): return None
    if val <= 50:  return 3   # Deep Poverty
    if val <= 100: return 2   # Poverty
    if val <= 200: return 1   # Near Poverty
    return 0                  # Stable

df['poverty_risk_score'] = df['POVPIP'].apply(categorize_poverty_score)
df = df.dropna(subset=['poverty_risk_score'])
df['poverty_risk_score'] = df['poverty_risk_score'].astype(int)
df['binary_target'] = (df['poverty_risk_score'] > 0).astype(int)

print('Target distribution (4-class):')
print(df['poverty_risk_score'].value_counts().sort_index())

## 3) Base Variable Recreation

In [ ]:
def map_puma_to_region(puma):
    if pd.isna(puma): return 'Unknown'
    p = int(puma)
    if p < 1500:             return 'Northern CA/Sierras'
    if 1500 <= p < 3000:     return 'Bay Area'
    if 3000 <= p < 5000:     return 'Central Valley'
    if 5000 <= p < 7000:     return 'Central Coast'
    if 7000 <= p < 9000:     return 'Los Angeles'
    if 9000 <= p < 10000:    return 'Inland Empire'
    if p >= 10000:           return 'San Diego/Border'
    return 'Unknown'

df['CA_Region'] = df['PUMA'].apply(map_puma_to_region)

## 4) Feature Engineering

> These engineered features were developed in the XGBoost workflow (Section 4). See Section 4 for full derivation rationale.

In [ ]:
#  Disability Severity Score (0-6) — replaces binary DIS
disability_cols = ['DEAR', 'DEYE', 'DPHY', 'DREM', 'DDRS', 'DOUT']
for col in disability_cols:
    df[f'{col}_binary'] = (df[col] == 1).astype(int)
df['disability_score'] = df[[f'{c}_binary' for c in disability_cols]].sum(axis=1)

#  Health Insurance — single binary (replaces HICOV, PRIVCOV, PUBCOV)
df['has_insurance'] = (df['HICOV'] == 1).astype(int)

#  Race / Ethnicity one-hot from raw binary flags (replaces RAC1P)
df['race_white']      = df['RACWHT']
df['race_black']      = df['RACBLK']
df['race_asian']      = df['RACASN']
df['race_indigenous'] = ((df['RACAIAN'] == 1) | (df['RACNH'] == 1) | (df['RACPI'] == 1)).astype(int)
df['race_other']      = df['RACSOR']
df['is_latinx']       = (df['HISP'] > 1).astype(int)
race_ohe_cols = ['race_white', 'race_black', 'race_asian', 'race_indigenous', 'race_other', 'is_latinx']
df['race_ethnic_aggregate'] = df[race_ohe_cols].sum(axis=1)

#  Education Tier (0-4) — replaces raw SCHL
def education_tier(schl):
    if pd.isna(schl) or schl <= 15: return 0  # No HS diploma
    if schl <= 17:  return 1                   # HS diploma / GED
    if schl <= 20:  return 2                   # Some college / Associate's
    if schl == 21:  return 3                   # Bachelor's degree
    return 4                                    # Advanced degree

df['education_tier'] = df['SCHL'].apply(education_tier)

## 5) Null Handling

In [ ]:
df['ENG']  = df['ENG'].fillna(0)
df['WKHP'] = df['WKHP'].fillna(0)
df['WRK']  = df['WRK'].fillna(0)
df['OCCP'] = df['OCCP'].fillna('NILF')

## 6) Train / Test Split

In [ ]:
df_train = df[df['year'] < 2024].copy()
df_test  = df[df['year'] == 2024].copy()

print(f'Train set: {len(df_train):,} rows')
print(f'Test set:  {len(df_test):,} rows')

## 7) Feature Sets

> **IMPORTANT:** We use the SAME `optimal_features` the XGBoost notebook selected via elbow analysis (k=12 parsimonious + 6 force-included race vars = 18). Keeping evaluation apples-to-apples between models.

> Top-12 predictive: `education_tier`, `MSP`, `WKHP`, `ESR`, `ENG`, `CIT`, `MAR`, `MIG`, `is_latinx`, `has_insurance`, `AGEP`, `OCCP`

> Force-included race: `race_white`, `race_black`, `race_asian`, `race_indigenous`, `race_other`, `race_ethnic_aggregate`

In [ ]:
engineered_features = [
    'disability_score', 'has_insurance',
    'race_white', 'race_black', 'race_asian', 'race_indigenous', 'race_other',
    'is_latinx', 'race_ethnic_aggregate', 'education_tier',
]
kept_features = [
    'AGEP', 'CIT', 'ENG', 'LANX', 'MAR', 'MIG', 'MSP', 'SEX',
    'NATIVITY', 'ESR', 'OCCP', 'WKHP', 'WKL', 'WRK', 'PUMA', 'CA_Region',
]
all_features = engineered_features + kept_features

numeric_features     = ['AGEP', 'WKHP', 'disability_score', 'education_tier', 'race_ethnic_aggregate']
binary_features      = ['has_insurance', 'race_white', 'race_black', 'race_asian',
                        'race_indigenous', 'race_other', 'is_latinx']
categorical_features = ['CIT', 'ENG', 'LANX', 'MAR', 'MIG', 'MSP', 'SEX',
                        'NATIVITY', 'ESR', 'OCCP', 'WKL', 'WRK', 'PUMA', 'CA_Region']

optimal_features = [
    # Predictive top-12 (elbow k=12)
    'education_tier', 'MSP', 'WKHP', 'ESR', 'ENG',
    'CIT', 'MAR', 'MIG', 'is_latinx', 'has_insurance', 'AGEP', 'OCCP',
    # Force-included race/ethnicity (domain justification)
    'race_white', 'race_black', 'race_asian',
    'race_indigenous', 'race_other', 'race_ethnic_aggregate',
]

optimal_categorical = [f for f in categorical_features if f in optimal_features]
optimal_binary      = [f for f in binary_features      if f in optimal_features]
optimal_numeric     = [f for f in numeric_features     if f in optimal_features]

print(f'Optimal feature set: {len(optimal_features)} features')
print(f'  Numeric:     {optimal_numeric}')
print(f'  Binary:      {optimal_binary}')
print(f'  Categorical: {optimal_categorical}')

## 8) OCCP Grouping & Label Encoding

In [ ]:
TOP_N_OCCP = 15
top_occp = df_train['OCCP'].value_counts().nlargest(TOP_N_OCCP).index.tolist()
top_occp_set = set(top_occp) | {'NILF'}

def group_occp(val):
    return val if val in top_occp_set else 'Other'

df_train['OCCP'] = df_train['OCCP'].apply(group_occp)
df_test['OCCP']  = df_test['OCCP'].apply(group_occp)
print(f'OCCP groups retained: {sorted(str(v) for v in top_occp_set | {"Other"})}')

In [ ]:
label_encoders = {}
for col in categorical_features:
    le = LabelEncoder()
    df_train[col] = df_train[col].astype(str)
    df_test[col]  = df_test[col].astype(str)
    le.fit(df_train[col])
    df_train[col] = le.transform(df_train[col])
    df_test[col] = df_test[col].apply(
        lambda x: le.transform([x])[0] if x in le.classes_ else -1
    )
    label_encoders[col] = le
print(f'Label encoded {len(categorical_features)} categorical features.')

## 9) Class Balancing (Downsampling)

In [ ]:
df_stable  = df_train[df_train['poverty_risk_score'] == 0]
df_at_risk = df_train[df_train['poverty_risk_score'] > 0]

df_stable_downsampled = resample(
    df_stable, replace=False, n_samples=len(df_at_risk), random_state=42
)

train_balanced = pd.concat([df_stable_downsampled, df_at_risk]).sample(frac=1, random_state=42)
print(f'Balanced class distribution:')
print(train_balanced['poverty_risk_score'].value_counts().sort_index())

## 10) Save Outputs — Logistic Regression

In [ ]:
train_balanced.to_csv('preprocessing_data/baseline_train_engineered.csv', index=False)
df_test.to_csv('preprocessing_data/baseline_test_engineered.csv', index=False)

print(f'Saved baseline_train_engineered.csv: {train_balanced.shape}')
print(f'Saved baseline_test_engineered.csv:  {df_test.shape}')
print(f'Models evaluate on {len(optimal_features)} features: {optimal_features}')

---
---

# SECTION 2 — Random Forest Preprocessing (Sparse Pipeline)

Reads the intermediate `train/test_data_final_feat_no_preprocessing.csv` files (produced by an earlier notebook), applies ACS-aware null handling, builds a sparse ColumnTransformer pipeline, and saves `.npz` sparse matrices + fitted preprocessor.

**Consumed by:** `4b_RandomForest_Model.ipynb` — sparse model section (cells 3-15).

---
---

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

## 1) Load Data

> These intermediate CSVs are produced by an earlier preprocessing notebook, not regenerated here.

In [ ]:
train_path = "preprocessing_data/train_data_final_feat_no_preprocessing.csv"
test_path  = "preprocessing_data/test_data_final_feat_no_preprocessing.csv"

df_train = pd.read_csv(train_path)
df_test  = pd.read_csv(test_path)

print("Train shape:", df_train.shape)
print("Test shape :", df_test.shape)
print("\nTrain target distribution:")
print(df_train["poverty_risk_score"].value_counts(dropna=False))

## 2) Null Exploration

In [ ]:
cols = ["WKHP","WKL","WRK","ENG","LANX","MIG","PUMA","POBP","OCCP","SCHL","ESR","CIT","MAR","MSP","NATIVITY","RAC1P","year"]
(df_train[cols].isna().mean().sort_values(ascending=False) * 100).round(2)

In [ ]:
df_train["ENG"].value_counts(dropna=False)

## 3) ACS-Aware NA Fixing

In [ ]:
def fix_acs_nas(df):
    df = df.copy()
    df["WKHP"] = df["WKHP"].replace({0: np.nan})
    df["WKL"]  = df["WKL"].replace({0: np.nan})
    df["WRK"]  = df["WRK"].map({1:1, 2:0, 0:np.nan})
    df["LANX"] = df["LANX"].map({1:1, 2:0, 0:np.nan})
    df.loc[(df["ENG"].isna()) & (df["LANX"] == 0), "ENG"] = 0
    df["MIG"]  = df["MIG"].replace({0: np.nan})
    df["ESR"]  = df["ESR"].replace({0: np.nan})
    df.loc[df["AGEP"] < 15, "MAR"] = np.nan
    df.loc[df["AGEP"] < 15, "MSP"] = np.nan
    for c in ["WKHP", "WKL", "ENG", "LANX", "MIG", "WRK"]:
        if c in df.columns:
            df[f"{c}_missing"] = df[c].isna().astype(int)
    return df

df_fixed = fix_acs_nas(df_train)
df_fixed["ENG_missing"].value_counts(dropna=False)
df_fixed["ENG"].isna().mean()
df_fixed["ENG"].value_counts(dropna=False).head(10)

'''ENG = 0 → speaks only English
ENG = 1–4 → proficiency
ENG_missing now represents true unexpected missingness'''

In [ ]:
df_train = fix_acs_nas(df_train)
df_test  = fix_acs_nas(df_test)
print("Train shape:", df_train.shape)
print("Test shape :", df_test.shape)
print("\nTrain target distribution:")
print(df_train["poverty_risk_score"].value_counts(dropna=False))

## 4) Preprocessing Function (Expanded Features)

In [ ]:
def preprocess_acs_data_rf(df: pd.DataFrame) -> pd.DataFrame:
    """Preprocess ACS-like features for Random Forest.
    - Do NOT fill binary columns with 0 (0 can be meaningful).
    - Map {1,2} survey binaries -> {1,0} while leaving NaN as NaN.
    - Keep full OCCP (no top-10 grouping).
    - Keep correlated features (RF doesn't require multicollinearity pruning).
    """
    df = df.copy()
    mapping_12 = {1: 1, 2: 0}

    for col in ["PRIVCOV", "PUBCOV", "DIS", "SEX"]:
        if col in df.columns:
            df[col] = df[col].map(mapping_12)

    if "HICOV" in df.columns:
        df["HICOV"] = df["HICOV"].map({1: 0, 2: 1})

    if "MAR" in df.columns:
        df["MAR"] = np.where(df["MAR"].isna(), np.nan, (df["MAR"] == 1).astype(int))

    if "CIT" in df.columns:
        df["CIT"] = np.where(df["CIT"].isna(), np.nan, (df["CIT"] < 5).astype(int))

    if "ESR" in df.columns:
        df["ESR_emp"] = np.where(df["ESR"].isna(), np.nan, df["ESR"].isin([1, 2]).astype(int))

    if "WRK" in df.columns:
        df["WRK"] = df["WRK"].map(mapping_12)

    if "MIG" in df.columns:
        df["MIG_recent"] = np.where(df["MIG"].isna(), np.nan, df["MIG"].isin([2, 3]).astype(int))

    if "NATIVITY" in df.columns:
        df["ForeignBorn"] = df["NATIVITY"].map({1: 0, 2: 1})

    if "POBP" in df.columns:
        df["Born_in_CA"] = np.where(df["POBP"].isna(), np.nan, (df["POBP"] == 6).astype(int))

    if "LANX" in df.columns:
        df["LANX"] = np.where(df["LANX"].isna(), np.nan, (df["LANX"] == 1).astype(int))

    if "SCHL" in df.columns:
        def recode_education(val):
            if pd.isna(val) or val == 0: return np.nan
            if val <= 15: return 0
            if val <= 17: return 1
            if val <= 20: return 2
            return 3
        df["SCHL_Tier"] = df["SCHL"].apply(recode_education)

    for cat_col in ["OCCP", "CA_Region", "RAC1P", "year", "MSP", "WKL", "ENG", "LANP", "PUMA", "POBP"]:
        if cat_col in df.columns:
            df[cat_col] = df[cat_col].astype("string")

    if "PUMA" in df.columns:
        df["PUMA"] = df["PUMA"].str.zfill(5)
    if "OCCP" in df.columns:
        mask = df["OCCP"].str.fullmatch(r"\d+")
        df.loc[mask, "OCCP"] = df.loc[mask, "OCCP"].str.zfill(4)
    if "POBP" in df.columns:
        df["POBP"] = df["POBP"].str.zfill(3)

    return df

In [ ]:
df_train_pp = preprocess_acs_data_rf(df_train)
df_test_pp  = preprocess_acs_data_rf(df_test)
print("Done preprocessing.")

In [ ]:
sub = df_train_pp.loc[df_train_pp["LANX"] == 0, "ENG"].astype("string")
print("ENG dtype:", sub.dtype)
print("Share ENG == '0.0' among LANX == 0:", (sub == "0.0").mean())
print("\nValue counts:")
print(sub.value_counts(dropna=False).head(10))

## 5) Feature Set (Expanded)

In [ ]:
features = [
    "AGEP", "WKHP", "SCHL_Tier",
    "SEX", "DIS", "CIT", "MAR", "WRK", "Born_in_CA", "LANX", "ForeignBorn", "ESR_emp", "MIG_recent",
    "MSP", "WKL", "ENG", "LANP",
    "OCCP", "RAC1P", "CA_Region",
    "PUMA", "POBP",
    "year"
]

target_col = "poverty_risk_score"
missing = [c for c in features + [target_col] if c not in df_train_pp.columns]
if missing:
    print("WARNING missing columns:", missing)

num_features = ["AGEP", "WKHP", "SCHL_Tier"]
bin_features = ["SEX","DIS","CIT","MAR","WRK","Born_in_CA","LANX","ForeignBorn","ESR_emp","MIG_recent"]
cat_features = ["MSP","WKL","ENG","LANP","OCCP","RAC1P","CA_Region","PUMA","POBP","year"]

X_train = df_train_pp[features].copy()
X_test  = df_test_pp[features].copy()
y_train = df_train_pp[target_col].astype(int)
y_test  = df_test_pp[target_col].astype(int)
print("X_train raw:", X_train.shape, "X_test raw:", X_test.shape)

## 6) Fix Pandas Nullable Types & Build Sparse Preprocessor

In [ ]:
for c in num_features + bin_features:
    X_train[c] = pd.to_numeric(X_train[c], errors="coerce")
    X_test[c]  = pd.to_numeric(X_test[c], errors="coerce")

for c in cat_features:
    X_train[c] = X_train[c].astype("object")
    X_test[c]  = X_test[c].astype("object")
    X_train[c] = X_train[c].where(pd.notna(X_train[c]), None)
    X_test[c]  = X_test[c].where(pd.notna(X_test[c]), None)

MIN_FREQ = 25

preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_features),
        ("bin", SimpleImputer(strategy="most_frequent"), bin_features),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=True, min_frequency=MIN_FREQ))
        ]), cat_features),
    ],
    remainder="drop"
)

X_train_sparse = preprocess.fit_transform(X_train)
X_test_sparse  = preprocess.transform(X_test)
print("X_train_sparse:", X_train_sparse.shape, type(X_train_sparse))
print("X_test_sparse :", X_test_sparse.shape, type(X_test_sparse))

## 7) Save Outputs — Random Forest Sparse

In [ ]:
import joblib
from scipy import sparse
from pathlib import Path

output_dir = Path("preprocessing_data")
output_dir.mkdir(exist_ok=True)

sparse.save_npz(output_dir / "X_train_sparse_rf.npz", X_train_sparse)
sparse.save_npz(output_dir / "X_test_sparse_rf.npz", X_test_sparse)
y_train.to_csv(output_dir / "y_train_rf.csv", index=False)
y_test.to_csv(output_dir / "y_test_rf.csv", index=False)
joblib.dump(preprocess, output_dir / "rf_preprocessor.joblib")

print("Saved: X_train/test_sparse_rf.npz, y_train/test_rf.csv, rf_preprocessor.joblib")

---
---

# SECTION 3 — XGBoost Preprocessing (Part 1): Load Raw Data & Recreate Base Variables

Loads the raw CSV (memory-efficient subset), filters to adults, creates poverty target, and recreates CA_Region. Feature engineering follows in Section 4.

**Consumed by:** `4c_XGBoost_Model.ipynb` (via `train_engineered.csv` produced at end of Section 4) and also `4b_RandomForest_Model.ipynb` engineered sections.

---
---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    precision_recall_curve, balanced_accuracy_score, ConfusionMatrixDisplay,
    recall_score
)
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

from imblearn.under_sampling import RandomUnderSampler
import xgboost as xgb
from collections import Counter

pd.set_option('display.max_columns', 50)
print('Libraries loaded.')

## 1) Load Raw Data (Memory-Efficient Subset)

In [ ]:
# Memory-efficient: only pull columns needed for feature engineering
# COW and SCH are included here because 4c_XGBoost_Model cell 44 uses them
usecols = [
    "year", "PUMA", "POVPIP", "AGEP",
    "SCHL", "HISP", "RAC1P",
    "RACWHT", "RACBLK", "RACASN", "RACAIAN", "RACNH", "RACPI", "RACSOR",
    "HICOV", "PRIVCOV", "PUBCOV",
    "DEAR", "DEYE", "DREM", "DPHY", "DDRS", "DOUT",
    "ENG", "LANX", "ESR", "WRK", "WKHP", "OCCP",
    "DIS", "CIT", "MAR", "MSP", "NATIVITY", "SEX", "MIG", "WKL",
    # Required by 4c_XGBoost_Model cell 44 engineer_features()
    "COW", "SCH",
]

path = "../1_Raw_Data/data_persons_ca_1yr/persons_master.csv"

df_raw = pd.read_csv(
    path,
    usecols=lambda c: c in set(usecols),
    low_memory=False
)

print(f"Raw (subset) shape: {df_raw.shape}")
print(f"Years present: {sorted(df_raw['year'].unique())}")
print(f"Age range: {df_raw['AGEP'].min()} - {df_raw['AGEP'].max()}")

In [ ]:
# Less memory-safe alternative — loads all 274 columns
#df_raw = pd.read_csv('../1_Raw_Data/data_persons_ca_1yr/persons_master.csv')
#print(f'Raw data shape: {df_raw.shape}')

## 2) Filter to Adults & Create Target

In [ ]:
df = df_raw[df_raw['AGEP'] > 18].copy()
print(f'Adults only: {df.shape}')

In [ ]:
def categorize_poverty_score(val):
    if pd.isna(val): return None
    if val <= 50: return 3      # Deep Poverty
    if val <= 100: return 2     # Poverty
    if val <= 200: return 1     # Near Poverty
    return 0                    # Stable

df['poverty_risk_score'] = df['POVPIP'].apply(categorize_poverty_score)
df = df.dropna(subset=['poverty_risk_score'])
df['poverty_risk_score'] = df['poverty_risk_score'].astype(int)
df['binary_target'] = (df['poverty_risk_score'] > 0).astype(int)

print('Target distribution (4-class):')
print(df['poverty_risk_score'].value_counts().sort_index())
print(f'\nBinary target distribution:')
print(df['binary_target'].value_counts())

In [ ]:
def map_puma_to_region(puma):
    if pd.isna(puma): return 'Unknown'
    p = int(puma)
    if p < 1500: return 'Northern CA/Sierras'
    if 1500 <= p < 3000: return 'Bay Area'
    if 3000 <= p < 5000: return 'Central Valley'
    if 5000 <= p < 7000: return 'Central Coast'
    if 7000 <= p < 9000: return 'Los Angeles'
    if 9000 <= p < 10000: return 'Inland Empire'
    if p >= 10000: return 'San Diego/Border'
    return 'Unknown'

df['CA_Region'] = df['PUMA'].apply(map_puma_to_region)
print('Region distribution:')
print(df['CA_Region'].value_counts())

---
---

# SECTION 4 — Final Feature Engineering (XGBoost-Derived, Used Across All Models)

⚠️  The feature engineering below was developed within the XGBoost preprocessing workflow.
The engineered variables — disability score, insurance binary, race/ethnicity encoding, and education tiers — are **shared across all three models** to ensure consistent, apples-to-apples evaluation.

This section also performs dimensionality reduction (XGBoost feature importance + elbow analysis) to select the `optimal_features` list used by the Logistic Regression baseline.

**Outputs:** `train_engineered.csv`, `test_engineered.csv`, `feature_engineering_metadata.json`

**Consumed by:** `4b_RandomForest_Model.ipynb` (engineered sections) · `4c_XGBoost_Model.ipynb`

---
---

## 1) Disability Aggregate

Sum of 6 individual disability indicators → severity score (0-6)

In [ ]:
disability_cols = ['DEAR', 'DEYE', 'DPHY', 'DREM', 'DDRS', 'DOUT']
for col in disability_cols:
    print(f'{col}: {df[col].value_counts().sort_index().to_dict()}')

In [ ]:
for col in disability_cols:
    df[f'{col}_binary'] = (df[col] == 1).astype(int)
df['disability_score'] = df[[f'{c}_binary' for c in disability_cols]].sum(axis=1)

print('Disability severity score distribution:')
print(df['disability_score'].value_counts().sort_index())
print(f'Correlation with poverty_risk_score: {df["disability_score"].corr(df["poverty_risk_score"]):.4f}')
print(f'Correlation with POVPIP: {df["disability_score"].corr(df["POVPIP"]):.4f}')
print(f'Original DIS correlation with POVPIP: {(df["DIS"] == 1).astype(int).corr(df["POVPIP"]):.4f}')

## 2) Health Insurance Collapse

Single binary: has any insurance (1) vs. no insurance (0)

In California, Medi-Cal qualification requires being at or below the poverty line. Keeping PRIVCOV/PUBCOV separately may introduce quasi-data-leakage since public insurance status is essentially a proxy for poverty status.

In [ ]:
print('HICOV (any insurance):', df['HICOV'].value_counts().sort_index().to_dict())
print('PRIVCOV (private):', df['PRIVCOV'].value_counts().sort_index().to_dict())
print('PUBCOV (public):', df['PUBCOV'].value_counts().sort_index().to_dict())

In [ ]:
df['has_insurance'] = (df['HICOV'] == 1).astype(int)
print('has_insurance distribution:')
print(df['has_insurance'].value_counts())
print(f'Correlation with POVPIP: {df["has_insurance"].corr(df["POVPIP"]):.4f}')
print(f'has_insurance (new):  {df["has_insurance"].corr(df["POVPIP"]):.4f}')
print(f'PRIVCOV (old):        {(df["PRIVCOV"] == 1).astype(int).corr(df["POVPIP"]):.4f}')
print(f'PUBCOV (old):         {(df["PUBCOV"] == 1).astype(int).corr(df["POVPIP"]):.4f}')
print(f'HICOV (old):          {(df["HICOV"] == 1).astype(int).corr(df["POVPIP"]):.4f}')

## 3) Race / Ethnicity Encoding

1. Collapse to 5 race categories: White, Black, Asian, Indigenous, Other
2. Add `HISP` as binary Latinx ethnicity indicator
3. One-hot encode the 5 race categories
4. Create racial-ethnic aggregate (sum → captures multiracial identities)

In [ ]:
race_binary_cols = ['RACWHT', 'RACBLK', 'RACASN', 'RACAIAN', 'RACNH', 'RACPI', 'RACSOR']
print('Individual race flag distributions:')
for col in race_binary_cols:
    print(f'  {col}: {df[col].value_counts().sort_index().to_dict()}')
print(f'HISP == 1 (Not Hispanic): {(df["HISP"] == 1).sum()}')
print(f'HISP > 1 (Hispanic/Latino): {(df["HISP"] > 1).sum()}')

In [ ]:
df['race_white'] = df['RACWHT']
df['race_black'] = df['RACBLK']
df['race_asian'] = df['RACASN']
df['race_indigenous'] = ((df['RACAIAN'] == 1) | (df['RACNH'] == 1) | (df['RACPI'] == 1)).astype(int)
df['race_other'] = df['RACSOR']
df['is_latinx'] = (df['HISP'] > 1).astype(int)
race_ohe_cols = ['race_white', 'race_black', 'race_asian', 'race_indigenous', 'race_other', 'is_latinx']
df['race_ethnic_aggregate'] = df[race_ohe_cols].sum(axis=1)

print('Race one-hot distribution:')
for col in race_ohe_cols:
    print(f'  {col}: {df[col].sum():,} ({df[col].mean()*100:.1f}%)')
print('Racial-ethnic aggregate distribution:')
print(df['race_ethnic_aggregate'].value_counts().sort_index())

In [ ]:
print('Correlation of race/ethnicity features with POVPIP:')
for col in race_ohe_cols + ['race_ethnic_aggregate']:
    corr = df[col].corr(df['POVPIP'])
    print(f'  {col}: {corr:.4f}')
print(f'Original RAC1P correlation: {df["RAC1P"].corr(df["POVPIP"]):.4f}')

## 4) Education Tiers

5-level education tier (0-4). ACS SCHL codes:
- 1–15: No HS diploma · 16–17: HS/GED · 18–20: Some college/Associate's · 21: Bachelor's · 22–24: Advanced

In [ ]:
def education_tier(schl):
    if pd.isna(schl) or schl <= 15: return 0
    if schl <= 17: return 1
    if schl <= 20: return 2
    if schl == 21: return 3
    return 4

df['education_tier'] = df['SCHL'].apply(education_tier)
tier_labels = {0: 'No HS', 1: 'HS/GED', 2: 'Some College/Assoc', 3: "Bachelor's", 4: 'Advanced'}
for tier, label in tier_labels.items():
    count = (df['education_tier'] == tier).sum()
    print(f'  {tier} ({label}): {count:,} ({count/len(df)*100:.1f}%)')
print(f'Correlation with POVPIP: {df["education_tier"].corr(df["POVPIP"]):.4f}')

## 5) Final Feature Set Definition

Defines `engineered_features`, `kept_features`, `all_features`, and `dropped_features`.

In [ ]:
engineered_features = [
    'disability_score', 'has_insurance',
    'race_white', 'race_black', 'race_asian', 'race_indigenous', 'race_other',
    'is_latinx', 'race_ethnic_aggregate', 'education_tier',
]
kept_features = [
    'AGEP', 'CIT', 'ENG', 'LANX', 'MAR', 'MIG', 'MSP', 'SEX',
    'NATIVITY', 'ESR', 'OCCP', 'WKHP', 'WKL', 'WRK', 'PUMA', 'CA_Region',
]
dropped_features = {
    'DIS':     'Replaced by disability_score (0-6)',
    'HICOV':   'Replaced by has_insurance (binary)',
    'PRIVCOV': 'Replaced by has_insurance — possible quasi-leakage',
    'PUBCOV':  'Replaced by has_insurance — Medi-Cal requires poverty-level income',
    'RAC1P':   'Replaced by one-hot race columns — label encoding was methodologically wrong',
    'SCHL':    'Replaced by education_tier (0-4)',
    'LANP':    'Dropped — 57% null, high cardinality (200+ languages)',
    'POBP':    'Dropped — high cardinality (200+ places of birth)',
    'year':    'Dropped — no temporal drift detected',
}
all_features = engineered_features + kept_features

print(f'Total features: {len(all_features)} ({len(engineered_features)} engineered + {len(kept_features)} kept)')
print(f'Dropped: {len(dropped_features)}')
for feat, reason in dropped_features.items():
    print(f'  {feat}: {reason}')

## 6) Null Handling & Preprocessing

In [ ]:
def analyze_nulls(df, name='Dataset'):
    null_counts = df.isnull().sum()
    null_pct = (null_counts / len(df) * 100).round(2)
    null_df = pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct})
    null_df = null_df[null_df['null_count'] > 0].sort_values('null_pct', ascending=False)
    print(f'=== {name} Null Analysis ===')
    print(null_df)
    return null_df

analyze_nulls(df[all_features], 'Engineered Feature Set')

### Understanding Null Patterns

Replace nulls with explicit ACS-compliant codes for interpretability.

In [ ]:
print('ENG null when LANX==2 (English only):',
      df[df['LANX'] == 2]['ENG'].isnull().mean() * 100, '%')
print('ENG null when LANX==1 (Other language):',
      df[df['LANX'] == 1]['ENG'].isnull().mean() * 100, '%')
print('OCCP null when ESR in [3,6] (Not employed):',
      df[df['ESR'].isin([3, 6])]['OCCP'].isnull().mean() * 100, '%')

In [ ]:
df['ENG']  = df['ENG'].fillna(0)
df['WKHP'] = df['WKHP'].fillna(0)
df['WRK']  = df['WRK'].fillna(0)
df['OCCP'] = df['OCCP'].fillna('NILF')

null_counts = df[all_features].isnull().sum()
null_counts = null_counts[null_counts > 0]
if len(null_counts) > 0:
    print('Remaining nulls:'); print(null_counts)
else:
    print('No null values in feature set.')

## 7) Train / Test Split

In [ ]:
train_mask = df['year'] < 2024
test_mask  = df['year'] == 2024

df_train = df[train_mask].copy()
df_test  = df[test_mask].copy()

print(f'Train set: {len(df_train):,} rows (years {sorted(df_train["year"].unique())})')
print(f'Test set:  {len(df_test):,} rows (year {df_test["year"].unique()})')
print(f'Train binary target: {df_train["binary_target"].value_counts().to_dict()}')
print(f'Test binary target:  {df_test["binary_target"].value_counts().to_dict()}')

## 8) Feature Classification & Label Encoding

In [ ]:
numeric_features     = ['AGEP', 'WKHP', 'disability_score', 'education_tier', 'race_ethnic_aggregate']
binary_features      = ['has_insurance', 'race_white', 'race_black', 'race_asian',
                        'race_indigenous', 'race_other', 'is_latinx']
categorical_features = ['CIT', 'ENG', 'LANX', 'MAR', 'MIG', 'MSP', 'SEX',
                        'NATIVITY', 'ESR', 'OCCP', 'WKL', 'WRK', 'PUMA', 'CA_Region']

print(f'Numeric ({len(numeric_features)}): {numeric_features}')
print(f'Binary ({len(binary_features)}): {binary_features}')
print(f'Categorical ({len(categorical_features)}): {categorical_features}')

In [ ]:
label_encoders = {}
for col in categorical_features:
    le = LabelEncoder()
    df_train[col] = df_train[col].astype(str)
    df_test[col]  = df_test[col].astype(str)
    le.fit(df_train[col])
    df_train[col] = le.transform(df_train[col])
    df_test[col]  = df_test[col].apply(
        lambda x: le.transform([x])[0] if x in le.classes_ else -1
    )
    label_encoders[col] = le

print(f'Label encoded {len(categorical_features)} categorical features.')
print(f'Train shape: {df_train[all_features].shape}')
print(f'Test shape:  {df_test[all_features].shape}')

In [ ]:
X_train = df_train[all_features].copy()
y_train = df_train['binary_target'].copy()
X_test  = df_test[all_features].copy()
y_test  = df_test['binary_target'].copy()

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_test:  {X_test.shape}, y_test:  {y_test.shape}')
print(f'y_train distribution: {Counter(y_train)}')
print(f'y_test distribution:  {Counter(y_test)}')

## 9) Dimensionality Reduction

1. XGBoost feature importance to rank features
2. Elbow plot — incremental feature addition by importance
3. K-Means silhouette analysis on feature space
4. Select `optimal_features` (parsimonious k + force-include race vars)

In [ ]:
rus = RandomUnderSampler(random_state=42)
X_train_us, y_train_us = rus.fit_resample(X_train, y_train)
print(f'Undersampled training set: {Counter(y_train_us)}')

quick_model = xgb.XGBClassifier(
    objective='binary:logistic', max_depth=6, learning_rate=0.1,
    n_estimators=200, subsample=0.8, colsample_bytree=0.8,
    random_state=42, eval_metric='logloss', early_stopping_rounds=20
)
quick_model.fit(X_train_us, y_train_us, eval_set=[(X_test, y_test)], verbose=False)

importance = pd.Series(
    quick_model.feature_importances_, index=all_features
).sort_values(ascending=False)

print('Feature Importance Ranking:')
for i, (feat, imp) in enumerate(importance.items(), 1):
    print(f'  {i:2d}. {feat:25s} {imp:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
importance.plot(kind='barh', ax=ax)
ax.set_xlabel('Feature Importance (Gain)')
ax.set_title('XGBoost Feature Importance — Engineered Features')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../3_Data_Preprocessing/preprocessing_data/feature_importance_engineered.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
feature_ranking = importance.index.tolist()
results_by_k = []

for k in range(2, len(feature_ranking) + 1):
    top_k = feature_ranking[:k]
    X_tr_k = X_train_us[top_k]
    X_te_k = X_test[top_k]
    model_k = xgb.XGBClassifier(
        objective='binary:logistic', max_depth=6, learning_rate=0.1,
        n_estimators=200, subsample=0.8, colsample_bytree=0.8,
        random_state=42, eval_metric='logloss', early_stopping_rounds=20
    )
    model_k.fit(X_tr_k, y_train_us, eval_set=[(X_te_k, y_test)], verbose=False)
    y_pred_k = model_k.predict(X_te_k)
    f1_k = f1_score(y_test, y_pred_k, average='macro')
    bal_acc_k = balanced_accuracy_score(y_test, y_pred_k)
    results_by_k.append({'k': k, 'macro_f1': f1_k, 'balanced_acc': bal_acc_k, 'features': top_k})
    print(f'  k={k:2d}: Macro F1={f1_k:.4f}, Balanced Acc={bal_acc_k:.4f} | +{feature_ranking[k-1]}')

results_df = pd.DataFrame(results_by_k)

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.plot(results_df['k'], results_df['macro_f1'], 'b-o', label='Macro F1', linewidth=2)
ax1.set_xlabel('Number of Features (k)', fontsize=12)
ax1.set_ylabel('Macro F1 Score', color='b', fontsize=12)
ax2 = ax1.twinx()
ax2.plot(results_df['k'], results_df['balanced_acc'], 'r--s', label='Balanced Accuracy', linewidth=2)
ax2.set_ylabel('Balanced Accuracy', color='r', fontsize=12)
f1_gains = results_df['macro_f1'].diff()
elbow_idx = f1_gains[f1_gains < 0.005].first_valid_index()
if elbow_idx is not None:
    elbow_k = results_df.loc[elbow_idx, 'k']
    ax1.axvline(x=elbow_k, color='green', linestyle=':', linewidth=2, label=f'Elbow at k={elbow_k}')
ax1.legend(loc='lower right')
ax2.legend(loc='upper left')
plt.title('Elbow Plot: Feature Count vs Model Performance', fontsize=14)
plt.tight_layout()
plt.savefig('../3_Data_Preprocessing/preprocessing_data/elbow_plot_features.png', dpi=150, bbox_inches='tight')
plt.show()
if elbow_idx is not None:
    print(f'Elbow detected at k={elbow_k} features')

In [ ]:
scaler = StandardScaler()
sample_idx = np.random.RandomState(42).choice(len(X_train_us), size=min(50000, len(X_train_us)), replace=False)
X_sample = scaler.fit_transform(X_train_us.iloc[sample_idx])

silhouette_scores = []
inertias = []
K_range = range(2, 15)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=100)
    labels = kmeans.fit_predict(X_sample)
    sil = silhouette_score(X_sample, labels, sample_size=10000, random_state=42)
    silhouette_scores.append(sil)
    inertias.append(kmeans.inertia_)
    print(f'  K={k:2d}: Silhouette={sil:.4f}, Inertia={kmeans.inertia_:.0f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(list(K_range), inertias, 'b-o')
ax1.set_xlabel('K'); ax1.set_ylabel('Inertia'); ax1.set_title('K-Means Elbow Plot')
ax2.plot(list(K_range), silhouette_scores, 'r-o')
ax2.set_xlabel('K'); ax2.set_ylabel('Silhouette Score'); ax2.set_title('K-Means Silhouette Plot')
plt.tight_layout()
plt.savefig('../3_Data_Preprocessing/preprocessing_data/kmeans_elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
best_row = results_df.loc[results_df['macro_f1'].idxmax()]
best_k = int(best_row['k'])
best_f1 = best_row['macro_f1']

threshold = best_f1 - 0.01
parsimonious = results_df[results_df['macro_f1'] >= threshold].iloc[0]
optimal_k = int(parsimonious['k'])
optimal_features = feature_ranking[:optimal_k]

print(f'Best k={best_k} with Macro F1={best_f1:.4f}')
print(f'Most parsimonious k={optimal_k} (within 0.01 of best)')

# Force-include all race/ethnicity variables (domain justification)
race_vars = ['race_white', 'race_black', 'race_asian', 'race_indigenous',
             'race_other', 'is_latinx', 'race_ethnic_aggregate']
added_race = [r for r in race_vars if r not in optimal_features]
optimal_features = optimal_features + added_race

print(f'Force-included {len(added_race)} race/ethnicity variables: {added_race}')
print(f'\nFinal feature set ({len(optimal_features)} features):')
for i, f in enumerate(optimal_features, 1):
    marker = ' (race — domain-included)' if f in added_race else ''
    print(f'  {i}. {f}{marker}')

## 10) Save Outputs — XGBoost & RF Engineered

In [ ]:
import json as _json

df_train.to_csv('preprocessing_data/train_engineered.csv', index=False)
df_test.to_csv('preprocessing_data/test_engineered.csv', index=False)

metadata = {
    'all_features': all_features,
    'engineered_features': engineered_features,
    'kept_features': kept_features,
    'numeric_features': numeric_features,
    'binary_features': binary_features,
    'categorical_features': categorical_features,
    'label_encoder_classes': {col: le.classes_.tolist() for col, le in label_encoders.items()},
    'optimal_features': optimal_features
}
with open('preprocessing_data/feature_engineering_metadata.json', 'w') as f:
    _json.dump(metadata, f, indent=2)

print(f'Saved train_engineered.csv: {df_train.shape}')
print(f'Saved test_engineered.csv:  {df_test.shape}')
print(f'Saved feature_engineering_metadata.json')
print(f'Columns in saved CSVs include COW: {"COW" in df_train.columns}')
print(f'Columns in saved CSVs include SCH: {"SCH" in df_train.columns}')